# Análise RFM (Recência, Frequência e Monetário)

Este notebook realiza a Análise RFM dos clientes a partir do conjunto de dados de e-commerce brasileiro da Olist.

## 1. Carregando os Dados

Carregamos os datasets de clientes (`olist_customers_dataset.csv`), pedidos (`olist_orders_dataset.csv`) e pagamentos (`olist_order_payments_dataset.csv`) a partir do diretório `data/raw/`.

In [1]:
import pandas as pd
import os

# Carregando os conjuntos de dados brutos
customers_df = pd.read_csv('../data/raw/olist_customers_dataset.csv')
orders_df = pd.read_csv('../data/raw/olist_orders_dataset.csv')
payments_df = pd.read_csv('../data/raw/olist_order_payments_dataset.csv')

print(f"Clientes: {customers_df.shape}")
print(f"Pedidos: {orders_df.shape}")
print(f"Pagamentos: {payments_df.shape}")

Clientes: (99441, 5)
Pedidos: (99441, 8)
Pagamentos: (103886, 5)


## 2. Cruzando e Filtrando os Datasets

Realizamos o merge dos três datasets e filtramos apenas os pedidos com status `'delivered'` (entregues), pois apenas vendas concluídas contam para a análise RFM padrão.

In [2]:
# Mesclar os pedidos com as informações do cliente
merged_df = orders_df.merge(customers_df, on='customer_id')

# Mesclar com os dados de pagamento
merged_df = merged_df.merge(payments_df, on='order_id')

# Filtrar apenas pedidos entregues
delivered_df = merged_df[merged_df['order_status'] == 'delivered'].copy()

print(f"Dataset após merge e filtro de entregues: {delivered_df.shape}")

Dataset após merge e filtro de entregues: (100756, 16)


## 3. Preparação das Datas

Convertemos a coluna `order_purchase_timestamp` para o tipo `datetime` para realizar os cálculos temporais da Recência.

In [3]:
# Converter data de compra para datetime
delivered_df['order_purchase_timestamp'] = pd.to_datetime(delivered_df['order_purchase_timestamp'])

## 4. Cálculo das Métricas RFM

Agrupamos os dados por `customer_unique_id` para calcular:
- **Recência (R)**: Diferença em dias entre a data da última compra do cliente e a data máxima de compra de todo o dataset.
- **Frequência (F)**: Contagem de pedidos únicos (`order_id`) por cliente.
- **Monetário (M)**: Soma do valor de pagamento (`payment_value`) por cliente.

In [4]:
# Data de referência para cálculo da recência (data máxima do dataset)
max_date = delivered_df['order_purchase_timestamp'].max()
print(f"Data máxima de compra no dataset: {max_date}")

# Agrupamento para cálculo das métricas RFM
rfm_df = delivered_df.groupby('customer_unique_id').agg(
    recency=('order_purchase_timestamp', lambda x: (max_date - x.max()).days),
    frequency=('order_id', 'nunique'),
    monetary=('payment_value', 'sum')
).reset_index()

print(f"Shape da matriz RFM: {rfm_df.shape}")
print(rfm_df.head())

Data máxima de compra no dataset: 2018-08-29 15:00:37
Shape da matriz RFM: (93357, 4)
                 customer_unique_id  recency  frequency  monetary
0  0000366f3b9a7992bf8c76cfdf3221e2      111          1    141.90
1  0000b849f77a49e4a4ce2b2a4ca5be3f      114          1     27.19
2  0000f46a3911fa3c0805444483337064      536          1     86.22
3  0000f6ccb0745a6a4b88665a16c9f078      320          1     43.62
4  0004aac84e0df4da2b147fca70cf8255      287          1    196.89


## 5. Salvando os Resultados

Exportamos o DataFrame resultante para `data/processed/rfm_data.csv`.

In [5]:
# Garantir a existência do diretório de saída
os.makedirs('../data/processed', exist_ok=True)

# Salvar matriz RFM resultante em CSV
output_path = '../data/processed/rfm_data.csv'
rfm_df.to_csv(output_path, index=False)
print(f"Matriz RFM salva com sucesso em: {output_path}")

Matriz RFM salva com sucesso em: ../data/processed/rfm_data.csv
